# 测试 `relay.qnn.simulated_dequantize`

In [ ]:
import tvm
from tvm import te
import numpy as np
from tvm import relay
from tvm.contrib import graph_executor
from tvm.runtime.vm import VirtualMachine
from tvm.topi.nn.qnn import SQNN_DTYPE_TO_CODE


def dequantize_test_driver(in_dtype, quant_args, axis, in_data):
    """
    QNN反量化测试驱动函数

    该函数创建一个Relay计算图，对输入数据执行反量化操作，并返回反量化结果。
    主要用于生成参考结果，以便与模拟反量化结果进行比较。

    参数:
    in_dtype: str - 输入数据类型
    quant_args: dict - 量化参数，包含in_zero_point和in_scale
    axis: int - 量化轴
    in_data: numpy.ndarray - 输入数据

    返回:
    numpy.ndarray - 反量化后的结果
    """
    # 获取输入数据形状
    shape = in_data.shape
    # 创建Relay变量
    input_data = relay.var("input_data", shape=shape, dtype=in_dtype)
    # 创建量化参数常量
    input_zero_point = relay.const(quant_args["in_zero_point"])
    input_scale = relay.const(quant_args["in_scale"])
    # 创建反量化操作
    dequantized_output = relay.qnn.dequantize(
        input_data,
        input_scale=input_scale,
        input_zero_point=input_zero_point,
        axis=axis,
    )
    # 创建Relay函数和IR模块
    mod = relay.Function(relay.analysis.free_vars(dequantized_output), dequantized_output)
    mod = tvm.IRModule.from_expr(mod)
    dev = tvm.cpu(0)
    target = "llvm"
    with tvm.transform.PassContext(opt_level=3):
        lib = relay.build(mod, target=target)
    # 创建运行时模块
    rt_mod = graph_executor.GraphModule(lib["default"](dev))
    rt_mod.set_input(input_data=in_data)
    rt_mod.run()
    # 获取输出结果
    res = rt_mod.get_output(0).numpy()
    return res


def build_simulated_dequantize(input_data, scale, zp, dtype, axis=-1):
    """
    构建QNN模拟反量化的虚拟机执行环境

    该函数创建一个包含模拟反量化操作的Relay计算图，编译它，并返回一个虚拟机实例，
    用于执行模拟反量化操作。

    参数:
    input_data: relay.Var - 输入数据变量
    scale: relay.Var - 缩放因子变量
    zp: relay.Var - 零点变量
    dtype: relay.Var - 数据类型变量
    axis: int - 量化轴，默认为-1（最后一个维度）

    返回:
    tvm.runtime.vm.VirtualMachine - 虚拟机实例
    """
    # 创建模拟反量化操作
    sim_q = relay.qnn.simulated_dequantize(
        input_data,
        scale,
        zp,
        axis=axis,
        in_dtype=dtype,
    )
    # 创建IR模块
    mod = tvm.IRModule.from_expr(sim_q)
    # 编译模型（优化级别3）
    with tvm.transform.PassContext(opt_level=3):
        vm_exec = relay.vm.compile(mod, "llvm", params=None)
    # 创建虚拟机实例
    vm = VirtualMachine(vm_exec, tvm.cpu(0))
    return vm


def verify_simulated_dequantize_simple(dtype):
    """
    验证简单QNN模拟反量化功能

    该函数通过比较常规反量化与模拟反量化的结果，验证模拟反量化功能的正确性。

    参数:
    dtype: str - 输入数据类型（"uint8", "int8", "int32"等）
    """
    # 生成随机测试数据
    data = np.random.uniform(low=-128, high=127, size=[2, 5]).astype(dtype)
    # 转换为float32用于模拟反量化
    data_fp = data.astype("float32")
    # 设置量化参数
    scale_np = np.float32(0.5)
    zp_np = np.int32(127)
    # 获取数据类型代码
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE[dtype])
    # 准备反量化测试参数
    quant_args = {"in_zero_point": zp_np, "in_scale": scale_np}
    # 执行常规反量化获取参考结果
    dq_out = dequantize_test_driver(
        in_dtype=dtype,
        quant_args=quant_args,
        axis=-1,
        in_data=data,
    )
    # 创建Relay变量
    input_data = relay.var("input_data", shape=data.shape, dtype="float32")
    scale = relay.var("scale", shape=[])
    zp = relay.var("zp", shape=[], dtype="int32")
    dtype = relay.var("dtype", shape=[], dtype="int32")
    # 构建模拟反量化虚拟机
    vm = build_simulated_dequantize(input_data, scale, zp, dtype)
    # 执行模拟反量化
    sim_dq_out = vm.invoke("main", input_data=data_fp, scale=scale_np, zp=zp_np, dtype=dtype_np)
    # 比较结果
    np.testing.assert_allclose(sim_dq_out.numpy(), dq_out, rtol=1e-5)


def test_simulated_dequantize():
    """
    QNN模拟反量化功能主测试函数

    该函数测试不同数据类型（uint8、int8、int32）的模拟反量化功能。
    """
    # 测试无符号8位整数反量化
    verify_simulated_dequantize_simple("uint8")
    # 测试有符号8位整数反量化
    verify_simulated_dequantize_simple("int8")
    # 测试有符号32位整数反量化
    verify_simulated_dequantize_simple("int32")


def test_dynamic_channels():
    """
    测试动态通道QNN模拟反量化功能

    该函数测试模拟反量化在不重新编译的情况下，支持标量和每通道量化参数的能力。
    先使用标量参数测试，然后不重新编译虚拟机，直接使用每通道参数测试。
    """
    # 生成随机测试数据
    data = np.random.uniform(low=-64, high=64, size=[2, 5]).astype("int8")
    data_fp = data.astype("float32")
    # 测试标量量化参数
    scale_np = np.asarray([0.5]).astype("float32")
    zp_np = np.asarray([0]).astype("int32")
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE["int8"])
    quant_args = {"in_zero_point": zp_np[0], "in_scale": scale_np[0]}
    # 获取标量参数反量化参考结果
    dq_out = dequantize_test_driver(
        in_dtype="int8",
        quant_args=quant_args,
        axis=0,
        in_data=data,
    )
    # 创建具有动态形状的Relay变量
    input_data = relay.var("input_data", shape=data.shape, dtype="float32")
    scale = relay.var("scale", shape=[relay.Any()], dtype="float32")
    zp = relay.var("zp", shape=[relay.Any()], dtype="int32")
    dtype = relay.var("dtype", shape=[], dtype="int32")
    # 构建模拟反量化虚拟机
    vm = build_simulated_dequantize(input_data, scale, zp, dtype, axis=0)
    # 使用标量参数执行模拟反量化
    sim_dq_out = vm.invoke("main", input_data=data_fp, scale=scale_np, zp=zp_np, dtype=dtype_np)
    np.testing.assert_allclose(sim_dq_out.numpy(), dq_out, rtol=1e-5)

    # 测试每通道量化参数（不重新编译虚拟机）
    scale_np = np.array([0.5, 0.25]).astype("float32")
    zp_np = np.array([127, 123]).astype("int32")

    # 获取每通道参数反量化参考结果
    quant_args = {"in_zero_point": zp_np, "in_scale": scale_np}
    dq_out = dequantize_test_driver(
        in_dtype="int8",
        quant_args=quant_args,
        axis=0,
        in_data=data,
    )
    # 不重新编译，直接使用每通道参数执行模拟反量化
    sim_dq_out = vm.invoke("main", input_data=data_fp, scale=scale_np, zp=zp_np, dtype=dtype_np)
    np.testing.assert_allclose(sim_dq_out.numpy(), dq_out, rtol=1e-5)


def test_dynamic_dtype():
    """
    测试动态数据类型QNN模拟反量化功能

    该函数测试模拟反量化在不重新编译的情况下，支持不同数据类型的能力。
    先使用uint8数据类型测试，然后不重新编译虚拟机，直接使用int8数据类型测试。
    """
    # 生成uint8随机测试数据
    data = np.random.uniform(low=0, high=255, size=[2, 5]).astype("uint8")
    data_fp = data.astype("float32")
    # 设置量化参数
    scale_np = np.asarray([0.5]).astype("float32")
    zp_np = np.asarray([127]).astype("int32")
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE["uint8"])
    quant_args = {"in_zero_point": zp_np[0], "in_scale": scale_np[0]}
    # 获取uint8反量化参考结果
    dq_out = dequantize_test_driver(
        in_dtype="uint8",
        quant_args=quant_args,
        axis=-1,
        in_data=data,
    )
    # 创建具有动态形状的Relay变量
    input_data = relay.var("input_data", shape=data.shape, dtype="float32")
    scale = relay.var("scale", shape=[relay.Any()], dtype="float32")
    zp = relay.var("zp", shape=[relay.Any()], dtype="int32")
    dtype = relay.var("dtype", shape=[], dtype="int32")
    # 构建模拟反量化虚拟机
    vm = build_simulated_dequantize(input_data, scale, zp, dtype)
    # 使用uint8数据类型执行模拟反量化
    sim_dq_out = vm.invoke("main", input_data=data_fp, scale=scale_np, zp=zp_np, dtype=dtype_np)
    np.testing.assert_allclose(sim_dq_out.numpy(), dq_out, rtol=1e-5)

    # 测试int8数据类型（不重新编译虚拟机）
    data = np.random.uniform(low=0, high=255, size=[2, 5]).astype("int8")
    data_fp = data.astype("float32")
    # 获取int8反量化参考结果
    dq_out = dequantize_test_driver(
        in_dtype="int8",
        quant_args=quant_args,
        axis=-1,
        in_data=data,
    )
    # 不重新编译，直接使用int8数据类型执行模拟反量化
    dtype_np = np.int32(SQNN_DTYPE_TO_CODE["int8"])
    sim_dq_out = vm.invoke("main", input_data=data_fp, scale=scale_np, zp=zp_np, dtype=dtype_np)
    np.testing.assert_allclose(sim_dq_out.numpy(), dq_out, rtol=1e-5)


if __name__ == "__main__":
    # 执行模拟反量化功能测试
    test_simulated_dequantize()
    # 执行动态通道测试
    test_dynamic_channels()
    # 执行动态数据类型测试
    test_dynamic_dtype()
